# 09 — Hyperparameter Tuning

## AI-Based NIFTY 50 Portfolio Risk Prediction & Early Warning System

Notebook 08 selected **Random Forest** as the candidate model using development data.

This notebook tunes that model without touching the final test period.

### Methodology
- Hyperparameter search uses **Train only**.
- `TimeSeriesSplit` preserves chronological order inside the training period.
- Validation remains a genuine out-of-sample model-selection checkpoint.
- The final test period is not used for tuning or threshold decisions.
- We compare tuned Random Forest against the Notebook 08 baseline.


## 1. Load data and recreate the chronological split

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    brier_score_loss,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

DATA = Path("../data/processed")
REPORTS = Path("../reports")
MODELS = Path("../models")

INPUT_PATH = DATA / "portfolio_ml_features.csv"
TARGET = "Target_DD_5Pct_10D"
RANDOM_STATE = 42

df = pd.read_csv(INPUT_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

target_columns = [
    "Target_DD_3Pct_10D",
    "Target_DD_5Pct_10D",
    "Target_DD_10Pct_10D",
]

excluded = {"Date", *target_columns}

feature_cols = [
    c for c in df.columns
    if c not in excluded
    and not c.startswith("Future_")
    and pd.api.types.is_numeric_dtype(df[c])
]

model_df = df.dropna(subset=[TARGET]).copy()
model_df[TARGET] = model_df[TARGET].astype(int)

X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan)
y = model_df[TARGET]

n = len(model_df)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train = model_df.iloc[:train_end].copy()
valid = model_df.iloc[train_end:valid_end].copy()
test = model_df.iloc[valid_end:].copy()

X_train = X.iloc[:train_end].copy()
X_valid = X.iloc[train_end:valid_end].copy()
X_test = X.iloc[valid_end:].copy()

y_train = y.iloc[:train_end].copy()
y_valid = y.iloc[train_end:valid_end].copy()
y_test = y.iloc[valid_end:].copy()

print("Rows:", len(model_df))
print("Features:", len(feature_cols))
print("Train:", train.Date.min(), "->", train.Date.max(), "|", len(train))
print("Validation:", valid.Date.min(), "->", valid.Date.max(), "|", len(valid))
print("Test:", test.Date.min(), "->", test.Date.max(), "|", len(test))
print("Train event rate:", y_train.mean())
print("Validation event rate:", y_valid.mean())
print("Test event rate:", y_test.mean())


## 2. Baseline Random Forest

This reproduces the Random Forest configuration used previously so tuning has a clear benchmark.


In [ ]:
baseline_params = {
    "n_estimators": 400,
    "max_depth": 6,
    "min_samples_leaf": 5,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(**baseline_params)),
])

baseline_model.fit(X_train, y_train)

baseline_valid_prob = baseline_model.predict_proba(X_valid)[:, 1]
baseline_valid_pred = (baseline_valid_prob >= 0.40).astype(int)

baseline_validation = {
    "Model": "Random_Forest_Baseline",
    "ROC_AUC": roc_auc_score(y_valid, baseline_valid_prob),
    "PR_AUC": average_precision_score(y_valid, baseline_valid_prob),
    "Precision_at_0.40": precision_score(y_valid, baseline_valid_pred, zero_division=0),
    "Recall_at_0.40": recall_score(y_valid, baseline_valid_pred, zero_division=0),
    "F1_at_0.40": f1_score(y_valid, baseline_valid_pred, zero_division=0),
    "Balanced_Accuracy_at_0.40": balanced_accuracy_score(y_valid, baseline_valid_pred),
    "Brier_Score": brier_score_loss(y_valid, baseline_valid_prob),
}

baseline_validation_df = pd.DataFrame([baseline_validation])
display(baseline_validation_df)


## 3. Time-aware cross-validation

The search is performed **only inside the training period**.

`TimeSeriesSplit` means each fold trains on earlier observations and validates on later observations.

No future training observations are allowed to influence an earlier fold.


In [ ]:
N_SPLITS = 4

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

cv_summary = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train), start=1):
    y_tr = y_train.iloc[tr_idx]
    y_va = y_train.iloc[va_idx]

    cv_summary.append({
        "Fold": fold,
        "Train_Rows": len(tr_idx),
        "Validation_Rows": len(va_idx),
        "Train_Start": train.iloc[tr_idx].Date.min(),
        "Train_End": train.iloc[tr_idx].Date.max(),
        "Fold_Validation_Start": train.iloc[va_idx].Date.min(),
        "Fold_Validation_End": train.iloc[va_idx].Date.max(),
        "Train_Event_Rate": y_tr.mean(),
        "Fold_Validation_Event_Rate": y_va.mean(),
        "Validation_Positive_Events": int(y_va.sum()),
    })

cv_summary_df = pd.DataFrame(cv_summary)
display(cv_summary_df)

assert all(
    row["Train_End"] < row["Fold_Validation_Start"]
    for _, row in cv_summary_df.iterrows()
)

print("TimeSeriesSplit chronology check: PASSED")


## 4. Hyperparameter search space

The search is deliberately moderate rather than enormous. The objective is to test whether reasonable regularization and tree complexity improve generalization.

Primary CV metric: **PR-AUC**.


In [ ]:
param_grid = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [4, 6, 8, None],
    "model__min_samples_leaf": [3, 5, 10],
    "model__max_features": ["sqrt", 0.7],
    "model__class_weight": ["balanced", "balanced_subsample"],
}

grid_size = len(list(ParameterGrid(param_grid)))

print("Hyperparameter combinations:", grid_size)


## 5. Manual time-series grid search

A manual loop is used instead of a generic randomized search so that every fold's PR-AUC can be inspected and failed/one-class folds can be handled safely.

Only **training-period observations** are used here.


In [ ]:
search_rows = []

for combo_number, params in enumerate(ParameterGrid(param_grid), start=1):
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train), start=1):
        y_tr = y_train.iloc[tr_idx]
        y_va = y_train.iloc[va_idx]

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1,
                **params,
            )),
        ])

        model.fit(X_train.iloc[tr_idx], y_tr)
        prob = model.predict_proba(X_train.iloc[va_idx])[:, 1]

        if y_va.nunique() < 2:
            pr_auc = np.nan
            roc_auc = np.nan
        else:
            pr_auc = average_precision_score(y_va, prob)
            roc_auc = roc_auc_score(y_va, prob)

        fold_scores.append(pr_auc)

        search_rows.append({
            "Combination": combo_number,
            "Fold": fold,
            "CV_PR_AUC": pr_auc,
            "CV_ROC_AUC": roc_auc,
            **params,
        })

search_results = pd.DataFrame(search_rows)

cv_results = (
    search_results
    .groupby(
        [
            "Combination",
            "model__n_estimators",
            "model__max_depth",
            "model__min_samples_leaf",
            "model__max_features",
            "model__class_weight",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        Mean_CV_PR_AUC=("CV_PR_AUC", "mean"),
        Std_CV_PR_AUC=("CV_PR_AUC", "std"),
        Mean_CV_ROC_AUC=("CV_ROC_AUC", "mean"),
        Valid_Folds=("CV_PR_AUC", "count"),
    )
    .sort_values(
        ["Mean_CV_PR_AUC", "Mean_CV_ROC_AUC"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(cv_results.head(10))


## 6. Select the best CV configuration

The winning configuration is selected from training-period cross-validation only.


In [ ]:
best_cv = cv_results.iloc[0].copy()

best_params = {
    "n_estimators": int(best_cv["model__n_estimators"]),
    "max_depth": (
        None
        if pd.isna(best_cv["model__max_depth"])
        else int(best_cv["model__max_depth"])
    ),
    "min_samples_leaf": int(best_cv["model__min_samples_leaf"]),
    "max_features": best_cv["model__max_features"],
    "class_weight": best_cv["model__class_weight"],
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

print("Best CV parameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

print("Mean CV PR-AUC:", best_cv["Mean_CV_PR_AUC"])
print("Mean CV ROC-AUC:", best_cv["Mean_CV_ROC_AUC"])


## 7. Fit tuned model on Train and evaluate on Validation

This is the key checkpoint.

The tuned configuration was chosen using only training-period folds. The validation period is now used once as the external development-period comparison against the baseline.


In [ ]:
tuned_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(**best_params)),
])

tuned_model.fit(X_train, y_train)

tuned_valid_prob = tuned_model.predict_proba(X_valid)[:, 1]
tuned_valid_pred = (tuned_valid_prob >= 0.40).astype(int)

tuned_validation = {
    "Model": "Random_Forest_Tuned",
    "ROC_AUC": roc_auc_score(y_valid, tuned_valid_prob),
    "PR_AUC": average_precision_score(y_valid, tuned_valid_prob),
    "Precision_at_0.40": precision_score(y_valid, tuned_valid_pred, zero_division=0),
    "Recall_at_0.40": recall_score(y_valid, tuned_valid_pred, zero_division=0),
    "F1_at_0.40": f1_score(y_valid, tuned_valid_pred, zero_division=0),
    "Balanced_Accuracy_at_0.40": balanced_accuracy_score(y_valid, tuned_valid_pred),
    "Brier_Score": brier_score_loss(y_valid, tuned_valid_prob),
}

tuned_validation_df = pd.DataFrame([tuned_validation])
display(tuned_validation_df)


## 8. Baseline vs tuned comparison

Tuning is accepted only if it provides a meaningful development-period improvement.

A more complex model is **not automatically better**.


In [ ]:
comparison = pd.concat(
    [baseline_validation_df, tuned_validation_df],
    ignore_index=True,
)

comparison["PR_AUC_Improvement_vs_Baseline"] = (
    comparison["PR_AUC"] - baseline_validation_df.loc[0, "PR_AUC"]
)

comparison["ROC_AUC_Improvement_vs_Baseline"] = (
    comparison["ROC_AUC"] - baseline_validation_df.loc[0, "ROC_AUC"]
)

comparison["Brier_Improvement_vs_Baseline"] = (
    baseline_validation_df.loc[0, "Brier_Score"] - comparison["Brier_Score"]
)

display(comparison)

baseline_pr = float(baseline_validation_df.loc[0, "PR_AUC"])
tuned_pr = float(tuned_validation_df.loc[0, "PR_AUC"])

if tuned_pr >= baseline_pr:
    tuning_decision = "ACCEPT_TUNED"
    selected_model_name = "Random_Forest_Tuned"
    selected_model = tuned_model
    selected_prob = tuned_valid_prob
else:
    tuning_decision = "KEEP_BASELINE"
    selected_model_name = "Random_Forest_Baseline"
    selected_model = baseline_model
    selected_prob = baseline_valid_prob

print("Tuning decision:", tuning_decision)
print("Selected development model:", selected_model_name)


## 9. Validation threshold analysis for the selected model

The model choice is now fixed. We inspect thresholds on validation data to understand the operational alert trade-off.

The existing Notebook 08 threshold of **0.40** is retained as the reference threshold. We also report the F1-optimal threshold as a diagnostic.


In [ ]:
threshold_grid = np.arange(0.10, 0.91, 0.05)

threshold_rows = []

for threshold in threshold_grid:
    pred = (selected_prob >= threshold).astype(int)

    threshold_rows.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(y_valid, pred, zero_division=0),
        "Recall": recall_score(y_valid, pred, zero_division=0),
        "F1": f1_score(y_valid, pred, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_valid, pred),
        "Alerts": int(pred.sum()),
        "Alert_Rate": float(pred.mean()),
    })

selected_threshold_results = pd.DataFrame(threshold_rows)

f1_optimal_threshold = float(
    selected_threshold_results.loc[
        selected_threshold_results["F1"].idxmax(),
        "Threshold"
    ]
)

reference_threshold = 0.40

display(selected_threshold_results)

print("Reference threshold:", reference_threshold)
print("F1-optimal validation threshold:", f1_optimal_threshold)


## 10. Threshold trade-off plot

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    selected_threshold_results["Threshold"],
    selected_threshold_results["Precision"],
    marker="o",
    label="Precision",
)

plt.plot(
    selected_threshold_results["Threshold"],
    selected_threshold_results["Recall"],
    marker="o",
    label="Recall",
)

plt.plot(
    selected_threshold_results["Threshold"],
    selected_threshold_results["F1"],
    marker="o",
    label="F1",
)

plt.axvline(
    reference_threshold,
    linestyle="--",
    label="Reference threshold = 0.40",
)

plt.xlabel("Alert threshold")
plt.ylabel("Metric")
plt.title("Validation Threshold Trade-off")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Inspect CV stability of the winning configuration

A strong mean score with unstable folds is less convincing than a slightly lower but consistent score.


In [ ]:
winning_combination = int(best_cv["Combination"])

winning_folds = (
    search_results[
        search_results["Combination"] == winning_combination
    ][
        ["Fold", "CV_PR_AUC", "CV_ROC_AUC"]
    ]
    .sort_values("Fold")
    .reset_index(drop=True)
)

display(winning_folds)

print(
    "CV PR-AUC mean:",
    winning_folds["CV_PR_AUC"].mean()
)

print(
    "CV PR-AUC std:",
    winning_folds["CV_PR_AUC"].std()
)


## 12. Refit the selected model on Train + Validation

The model decision is now fixed using development data.

The final test set is still untouched by this fitting step and will be evaluated only in the designated final-evaluation workflow.


In [ ]:
train_valid = pd.concat([train, valid], ignore_index=True)

X_train_valid = train_valid[feature_cols].copy()
X_train_valid = X_train_valid.replace([np.inf, -np.inf], np.nan)
y_train_valid = train_valid[TARGET].copy()

if tuning_decision == "ACCEPT_TUNED":
    final_selected_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(**best_params)),
    ])
else:
    final_selected_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(**baseline_params)),
    ])

final_selected_model.fit(X_train_valid, y_train_valid)

print("Final development refit observations:", len(train_valid))
print("Selected model:", selected_model_name)
print("Reference threshold:", reference_threshold)


## 13. Save tuning outputs and model artifact

In [ ]:
TUNING_RESULTS_PATH = REPORTS / "hyperparameter_search_results.csv"
TUNING_SUMMARY_PATH = REPORTS / "hyperparameter_tuning_summary.csv"
THRESHOLD_PATH = REPORTS / "tuned_model_threshold_analysis.csv"
CV_FOLD_PATH = REPORTS / "tuned_model_cv_folds.csv"
MODEL_PATH = MODELS / "09_tuned_selected_model.joblib"

search_results.to_csv(TUNING_RESULTS_PATH, index=False)
cv_results.to_csv(TUNING_SUMMARY_PATH, index=False)
selected_threshold_results.to_csv(THRESHOLD_PATH, index=False)
winning_folds.to_csv(CV_FOLD_PATH, index=False)

artifact = {
    "model_name": selected_model_name,
    "model": final_selected_model,
    "feature_columns": feature_cols,
    "target": TARGET,
    "reference_threshold": reference_threshold,
    "f1_optimal_validation_threshold": f1_optimal_threshold,
    "baseline_parameters": baseline_params,
    "tuned_parameters": best_params,
    "tuning_decision": tuning_decision,
    "selection_metric": "validation PR-AUC after TimeSeriesSplit tuning on training period",
    "random_state": RANDOM_STATE,
}

joblib.dump(artifact, MODEL_PATH)

print("Saved:")
print(TUNING_RESULTS_PATH)
print(TUNING_SUMMARY_PATH)
print(THRESHOLD_PATH)
print(CV_FOLD_PATH)
print(MODEL_PATH)


## 14. Final quality checks

In [ ]:
assert train["Date"].max() < valid["Date"].min() < test["Date"].min()

assert not any(
    ("future" in c.lower() or "target" in c.lower())
    for c in feature_cols
)

assert np.isinf(
    X_train_valid.select_dtypes(include=np.number)
).sum().sum() == 0

assert selected_model_name in {
    "Random_Forest_Baseline",
    "Random_Forest_Tuned",
}

assert 0 < reference_threshold < 1
assert 0 < f1_optimal_threshold < 1

assert (MODEL_PATH).exists()
assert len(cv_results) > 0
assert len(winning_folds) == N_SPLITS

print("Tuning decision:", tuning_decision)
print("Selected model:", selected_model_name)
print("Reference threshold:", reference_threshold)
print("F1-optimal validation threshold:", f1_optimal_threshold)
print("Baseline validation PR-AUC:", baseline_pr)
print("Selected validation PR-AUC:", float(
    tuned_validation_df.loc[0, "PR_AUC"]
    if tuning_decision == "ACCEPT_TUNED"
    else baseline_validation_df.loc[0, "PR_AUC"]
))
print("FINAL NOTEBOOK 09 QUALITY CHECKS: PASSED")


# Conclusion

Notebook 09 has now tested whether hyperparameter tuning improves the Random Forest using a **time-aware training-only cross-validation process**.

The final test period remains untouched.

### Next step
`10_shap_explainability.ipynb`

That notebook will explain which portfolio features drive the selected model's risk predictions and turn the model into an interpretable early-warning framework.
